In [8]:
from huggingface_hub import hf_hub_download
import pandas as pd

path = hf_hub_download(
    repo_id="project-riz/osu-beatmap-tags",
    filename="tags.csv",  # check exact name on the Files tab
    repo_type="dataset",
)

df = pd.read_csv(path)

In [ ]:
entries = []
for _, row in df.iterrows():
    beatmapset_id = row["beatmapset_id"]
    for col in df.columns:
        if col == "beatmapset_id":
            continue
        value = row[col]
        if pd.isna(value) or value == 0:
            continue
        entries.append({
            "beatmapset_id": int(beatmapset_id),
            "tag_name": col,
            "vote_count": int(value)
        })

entries

[{'beatmapset_id': np.float64(989587.0),
  'tag_name': 'streams/bursts',
  'vote_count': 1},
 {'beatmapset_id': np.float64(2594989.0),
  'tag_name': 'streams/bursts',
  'vote_count': 3},
 {'beatmapset_id': np.float64(2594989.0),
  'tag_name': 'expression/simple',
  'vote_count': 3},
 {'beatmapset_id': np.float64(2594990.0),
  'tag_name': 'streams/bursts',
  'vote_count': 1},
 {'beatmapset_id': np.float64(2594991.0),
  'tag_name': 'streams/bursts',
  'vote_count': 2},
 {'beatmapset_id': np.float64(2594991.0),
  'tag_name': 'tech/finger control',
  'vote_count': 1},
 {'beatmapset_id': np.float64(2594993.0),
  'tag_name': 'skillset/jumps',
  'vote_count': 2},
 {'beatmapset_id': np.float64(2594993.0),
  'tag_name': 'streams/bursts',
  'vote_count': 5},
 {'beatmapset_id': np.float64(2594993.0),
  'tag_name': 'jumps/wide',
  'vote_count': 5},
 {'beatmapset_id': np.float64(4045755.0),
  'tag_name': 'streams/bursts',
  'vote_count': 1},
 {'beatmapset_id': np.float64(4045755.0),
  'tag_name': '

In [10]:
import psycopg2
from psycopg2.extras import execute_values

conn = psycopg2.connect(
    host="localhost",
    dbname="beatmap_similarity",
    user="postgres",
    password=""
)

def insert_batch(conn, rows):
    cols = list(rows[0].keys())
    values = []
    for r in rows:
        row_val = []
        for c in cols:
            row_val.append(r[c])
        values.append(row_val)

    query = f"""
        insert into beatmapset_tags({','.join(cols)})
        values %s
        on conflict (beatmapset_id, tag_name) do nothing
    """

    with conn.cursor() as cur:
        execute_values(cur, query, values)
    conn.commit()

insert_batch(conn, entries)

InvalidSchemaName: schema "np" does not exist
LINE 3:         values (np.float64(989587.0),'streams/bursts',1),(np...
                        ^
